In [60]:
import scipy.io as sio
import numpy as np
import openslide
from src.histo_kit.utils.wsi import load_wsi_mag
from src.histo_kit.grand_qc.artifacts import Artifact
from src.histo_kit.utils.wsi import get_regions_location
from PIL import Image

In [61]:
# desired magnification for extraction
des_mag = 10

In [62]:
wsi_path = "/mnt/data/Datasets/Compass/HE/110-06_he_1.svs"
mask_qc = "/mnt/data/Tmp/jmerta/HE-masks-compass_30_11_2025/masks_grandqc/110-06_he.mat"
mask_bg = "/mnt/data/Tmp/jmerta/HE-masks-compass_30_11_2025/masks/110-06_he.mat"

In [63]:
wsi = openslide.OpenSlide(wsi_path)
mask_qc = sio.loadmat(mask_qc)
mask_bg = sio.loadmat(mask_bg)
region, scale_val, info, mpp_slide, ratio = load_wsi_mag(wsi, des_mag, allow_upscaling=True)

In [ ]:
des_h, des_w, _ = np.array(region).shape
bg = Image.fromarray(mask_bg["mask_bg"])
bg = np.array(bg.resize((des_w, des_h), resample=Image.NEAREST))

bbox, mask_list = get_regions_location(bg)



In [ ]:
print(mask_qc.keys())
#new_mask_qc = np.zeros_like(region[:,:,0:1])
scale_val = mask_qc["scale_val"][0][0]
scale_val_reg = des_mag / mask_qc["mag_l0"][0][0]
print(scale_val_reg)

In [ ]:
new_mask_qc = np.zeros_like(np.array(region)[:,:,0:1])
scale_val = mask_qc["scale_val"][0][0]
scale_val_reg = des_mag / mask_qc["mag_l0"][0][0]

# how much we have to scale the bboxes
scale_factor = scale_val_reg/scale_val
print(scale_factor)

new_bboxes = []
for bbox in mask_qc["bbox"]:
    y0, x0, y1, x1 = bbox*scale_factor
    n_b = [max(y0, 0), max(x0, 0), min(y1, new_mask_qc.shape[0]), min(x1, new_mask_qc.shape[1])]
    new_bboxes.append(n_b)

for mask in mask_qc["mask_art"][0]:
    print(mask.shape)

In [67]:
from src.histo_kit.patches_extraction.preprocessing import convert_mask_grandqc

mask_qc = "/mnt/data/Tmp/jmerta/HE-masks-compass_30_11_2025/masks_grandqc/110-06_he.mat"
mask_qc = sio.loadmat(mask_qc)
reg_list, full = convert_mask_grandqc(mask_qc, region, des_mag, art_exclude = [Artifact.BG_MODEL, Artifact.ART_FOLD], mode = "wsi")


KeyboardInterrupt: 